In [ ]:
import math
import copy
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from itertools import count
from importnb import Notebook
from typing import Any, List, Tuple, Dict

import os
import sys

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.PrefetchScheduler import PrefetchScheduler

import Common.config as config
import Common.datatypes as datatypes
import Common.utils as utils

import importlib

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(utils)

TransitionTuple = datatypes.TransitionTuple

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    metadata = {"render.modes": []}

    def __init__(
        self, 
        cfg: config.Config,
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: None,
        du_caches: None,
        mec_cache: None,
        latency_model: None,
        prefetch_fn: None,
        reward_fn: None,
        max_steps: int = 10000,
        *,
        step_duration_s=1.0,
        debugger=None
    ):
        super().__init__()

        self.cfg = cfg
        self.step_count = 0
        self.step_duration_s = step_duration_s
        self.max_steps = max_steps

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches
        self.mec_cache = mec_cache
        self.latency_model = latency_model

        self.prefetch_fn = prefetch_fn or (lambda cache, action: cache.drl_prefetching(action))
        self.reward_fn = reward_fn or (lambda info: info.get('reward_per_user', {}).get(info.get('current_user', -1), 0.0))

        # History holders
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }
        
        self.scheduler = PrefetchScheduler(
            R_M_D=self.latency_model.R_M_D,
            R_C_M=self.latency_model.R_C_M,
            U=self.latency_model.max_U,
            step_duration_s=self.step_duration_s
        )

        self.debugger = debugger
        
    # ─────────────────────────────────────────────────────────────────────────
    # Internal Helpers
    # ─────────────────────────────────────────────────────────────────────────
    def _make_ready_bitmaps(self, du_planned, mec_planned):
        du_ready = None
        mec_ready = None

        if du_planned:
            du_ready = []
            for du_idx, bm in enumerate(du_planned):
                cache_key = f"DU:{du_idx}"
                du_ready.append(self.scheduler.materialize_ready_bitmap(cache_key, bm))

        if mec_planned is not None:
            mec_ready = self.scheduler.materialize_ready_bitmap("MEC", mec_planned)

        return du_ready, mec_ready

    def _missing_items(self, req: Dict[str, Any]) -> list[int]:
        missing = 5 * [0]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx
        
        video = req["video"]
        viewport = req["viewport"]
        
        if video not in video_cache_index:
            missing[0] = 1  # Base layer missing
            missing[1:] = [1, 1, 1, 1]  # All enhancement layers missing
        else:
            video_idx = video_cache_index.index(video)
            cached_tiles = tile_cache_index[video_idx]

            missing[0] = 0  # Base layer present
            for idx, tile in enumerate(viewport):
                if tile not in cached_tiles:
                    missing[idx + 1] = 1  # Enhancement layer missing

        return missing

    def _create_transition(self, state, action_idx) -> TransitionTuple:
        """
        Create frozen DRL transition snapshot.
        Deep copy avoids mutation by stateful feature updates.
        """
        return TransitionTuple(
            state=copy.deepcopy(state),
            action=action_idx,
            reward=0.0,
            next_state=None
        )

    def _process_prefetch_actions(
        self,
        req,
        agent,
        net_adapter,
        du_plan,
        mec_plan
    ) -> Tuple[List[TransitionTuple], Any, Any]:

        user = req["u"]
        video = req["video"]
        viewport = req["viewport"]

        backhaul_usage = 0.0
        transitions = [None] * 5
        missing = self._missing_items(req)

        for idx, is_missing in enumerate(missing):
            if idx == 0:
                self.debugger.log(f'base_layer_missing', is_missing)
            else:
                self.debugger.log(f'enh_layer_missing_{idx}', is_missing)

        for idx, is_missing in enumerate(missing):
            
            if not is_missing: continue

            video_cache_idx = self.mec_cache.get_video_cache_idx(video)
            if idx > 0 and video_cache_idx == -1:
                break

            # Build state representation
            state = net_adapter.build_observation(
                video, viewport[idx - 1] if idx > 0 else None
            )

            # DRL action selection
            action_idx, _ = agent.select_action(state, idx, video_cache_idx)

            action = {
                "user": user,
                "video": video,
                "tiles": [] if idx == 0 else [viewport[idx - 1]],
                "base_req_init": (idx == 0),
                "action_idx": action_idx
            }
            mec_plan = self.prefetch_fn(self.mec_cache, action)

            # Create immutable transition snapshot
            transitions[idx] = self._create_transition(state, action_idx)

            if idx == 0:
                self.debugger.log('base_action', action_idx)

                if action_idx != 0 and video_cache_idx == -1:
                    backhaul_usage += 12 * self.mec_cache.tile_size_bytes[0]
            else:
                if action_idx != 0 and video_cache_idx != -1 and viewport[idx - 1] not in self.mec_cache.policy.tile_idx[video_cache_idx]:
                    backhaul_usage += self.mec_cache.tile_size_bytes[idx]
                                    
                self.debugger.log(f'enh_layer_{idx}_action', action_idx)

        self.debugger.log("backhaul_usage", backhaul_usage)

        return transitions, du_plan, mec_plan

    def _compute_cache_hits(self, req):
        video = req["video"]
        viewport = req["viewport"]

        video_cache_index = self.mec_cache.policy.video_idx
        tile_cache_index = self.mec_cache.policy.tile_idx

        base_hit = 1 if video in video_cache_index else 0
        
        video_idx = self.mec_cache.get_video_cache_idx(video)

        if video_idx != -1:
            enh_hit = [
                1 if tile in tile_cache_index[video_idx] else 0 for tile in viewport
            ]
        else:
            enh_hit = [0, 0, 0, 0]
    
        return dict(
            base_layer_hits=12 * base_hit,
            enh_layer_hits=sum(enh_hit),
            base_layer_misses=12 * (1 - base_hit),
            enh_layer_misses=4 - sum(enh_hit)
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Gym Environment API
    # ─────────────────────────────────────────────────────────────────────────
    def step(self, agent=None, net_adapter=None, req=None, cfg=None):
        info = {}

        # ------------------------------------------------------------
        # 1. Handle each user's missing tiles with DRL decisions
        # ------------------------------------------------------------
        transitions, _, _ = self._process_prefetch_actions(
            req, agent, net_adapter, None, None
        )

        nxt_req = self.users_env.get_next_request(None, None)

        reward_val = net_adapter.features.compute_reward(window_size=1)
        if nxt_req is None:
            nxt_req = req

        video = nxt_req["video"]
        viewport = nxt_req["viewport"]

        for idx, trans in enumerate(transitions):
            
            if trans is None: continue

            nxt_state = net_adapter.build_observation(
                video, viewport[idx - 1] if idx > 0 else None
            )

            trans.next_state = nxt_state
            transitions[idx].reward = reward_val

            agent.remember(
                transitions[idx].state,
                transitions[idx].action,
                transitions[idx].reward,
                transitions[idx].next_state,
                done=self.users_env.all_users_done()
            )
        
            agent.train_step()

        # -------------------------------------------------------
        # 3. Remaining users requests
        # -------------------------------------------------------
        info["user_request"] = nxt_req

        video = nxt_req["video"]
        viewport = nxt_req["viewport"]

        net_adapter.features.update_history(video, viewport)
        net_adapter.features.update_ch_history(video, viewport)

        # -------------------------------------------------------
        # 4. Cache stats (HIT / MISS)
        # -------------------------------------------------------
        info.update(self._compute_cache_hits(nxt_req))

        # -------------------------------------------------------
        # 5. Compute final per-user reward
        # -------------------------------------------------------
        reward = reward_val

        # -------------------------------------------------------
        # 6. Advance simulation time (1 step)
        # -------------------------------------------------------
        done = self.users_env.all_users_done() or (self.step_count >= self.max_steps - 1)
        self.step_count += 1

        return {}, reward, done, info


    # ---------------------------------------------------------
    # REWARD FUNCTION
    # ---------------------------------------------------------
    def compute_reward(self, psnr_per_user):
        reward_per_user = {}
        for u, psnr in psnr_per_user.items():    
            reward_per_user[u] = psnr

        return reward_per_user

    # ---------------------------------------------------------
    #  SAMPLE ACTION (for testing)
    # ---------------------------------------------------------
    def sample_action(self):
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            centers = [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]
            for x, y in centers:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }, c * self.n + c

    # ---------------------------------------------------------
    # WARM-UP PHASE
    # ---------------------------------------------------------
    def warmup_phase(self, net_adapter, num_steps=1000):
        """Warm-up phase to initialize cache with some videos."""

        base_idx_counter = 1
        enh_idx_counter = 1
        
        for _ in range(num_steps):  # Arbitrary number of warm-up steps
            req = self.users_env.get_next_request(None, None)
            video = req["video"]

            vid_idx = self.mec_cache.get_video_cache_idx(video)
            if vid_idx == -1:
                action_idx = base_idx_counter
                base_idx_counter = (base_idx_counter + 1) % self.cfg.cache_size
            else:
                action_idx = vid_idx

            action = {
                "video": video,
                "tiles": [],
                "base_req_init": True,
                "action_idx": action_idx # Dummy action index for warm-up
            }
            self.prefetch_fn(self.mec_cache, action)
            
            viewport = req["viewport"]
            for idx in range(4):

                if vid_idx != -1 and viewport[idx] in self.mec_cache.policy.tile_idx[vid_idx]:
                    continue

                tile_action = {
                    "video": video,
                    "tiles": [viewport[idx]],
                    "base_req_init": False,
                    "action_idx": self.cfg.cache_size + (action_idx - 1) * 4 + 1 + enh_idx_counter
                }
                self.prefetch_fn(self.mec_cache, tile_action)
                
                enh_idx_counter = (enh_idx_counter + 1) % self.cfg.viewport
            
            net_adapter.features.update_history(video, viewport)

    # ---------------------------------------------------------
    # RESET
    # ---------------------------------------------------------
    def reset(self, **kwargs):
        self.step_count = 0
        
        self.users_reward = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.users_psnr = {
            u: [] for u in range(self.users_env.n_users)
        }
        self.total_gop_requests_per_user = {
            u: 0 for u in range(self.users_env.n_users)
        }

        _, info_users = self.users_env.reset(**kwargs)

        info_cache_mec = self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {}

        req = self.users_env.get_next_request(None, None)

        info_cache = {
            **info_cache_mec,
            **info_users,
            "user_request": req
        }
        
        # Reset scheduler's time and availability
        self.scheduler.now_s = 0.0
        self.scheduler.availability = {}

        return None, info_cache